In [1]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json
from collections import Counter

input_path = "/content/drive/MyDrive/Joleop Project/data/train/finetune_dataset_no_bkl.jsonl"

labels = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        text = s["messages"][-1]["content"][0]["text"].lower().strip()
        labels.append(text)

print("📊 상위 30개 라벨 텍스트 예시:")
for t in labels[:30]:
    print("-", t)

print("\n📈 라벨 텍스트 빈도:")
for lbl, cnt in Counter(labels).most_common(20):
    print(f"{lbl}: {cnt}")


📊 상위 30개 라벨 텍스트 예시:
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반
- 멜라닌 세포모반

📈 라벨 텍스트 빈도:
멜라닌 세포모반: 7867
흑색종: 3036
지루각화증: 1625
아토피 피부염: 1119
습진: 299
여드름: 298
각화증: 298
정상피부: 124


In [5]:
import json
import random
import unicodedata
from collections import defaultdict

input_path = "/content/drive/MyDrive/Joleop Project/data/train/finetune_dataset_no_bkl.jsonl"
output_path = "/content/drive/MyDrive/Joleop Project/data/train/finetune_dataset_diverse_dynamic.jsonl"

random.seed()

# 질병명 매칭(한글·영문)
label_aliases = {
    "여드름": ["여드름", "acne"],
    "습진": ["습진", "eczema"],
    "아토피 피부염": ["아토피", "atopic_dermatitis"],
    "지루각화증": ["지루각화증", "seborrheic_keratosis"],
    "멜라닌 세포모반": ["멜라닌 세포모반", "nevus", "nevi"],
    "흑색종": ["흑색종", "흑색종", "melanoma"],
    "각화증": ["각화증", "keratosis"],
    "정상피부": ["정상피부", "normal_skin"]
}

# 증상·원인·조언 조합 데이터
symptom_parts = {
    "여드름": ["피부에 붉은 뾰루지가 보입니다", "모공이 막혀 피지가 쌓인 모습입니다", "피부 표면에 염증성 병변이 있습니다"],
    "습진": ["피부가 건조하고 가려운 증상이 있습니다", "각질이 일어나며 붉은 반점이 보입니다", "진물과 긁은 자국이 있습니다"],
    "아토피 피부염": ["피부가 두꺼워지고 붉게 염증이 생겼습니다", "가려움이 심하고 피부 장벽이 약해 보입니다", "만성적인 염증 패턴이 관찰됩니다"],
    "지루각화증": ["피부에 갈색 각질성 돌기가 있습니다", "노화로 인한 과각화성 병변으로 보입니다", "표면이 거칠고 돌출된 병변이 있습니다"],
    "멜라닌 세포모반": ["피부에 어두운 점이 있으며 경계가 뚜렷합니다", "색소가 고르게 분포되어 있습니다", "일반적인 모반 형태를 띱니다"],
    "흑색종": ["점의 색이 불균일하고 경계가 불명확합니다", "색소가 퍼진 듯한 모양입니다", "피부 아래 깊은 색소 침착이 보입니다"],
    "각화증": ["피부가 두꺼워지고 하얀 각질이 일어납니다", "각질층이 두껍게 형성되어 있습니다", "피부 표면이 거칠어 보입니다"],
    "정상피부": ["피부가 깨끗하고 염증이 없습니다", "색 변화나 병변이 없습니다", "매끄러운 피부결이 유지되고 있습니다"]
}

advice_parts = [
    "전문의의 진료를 권장드립니다.",
    "보습제를 자주 바르면 도움이 됩니다.",
    "손으로 자극하지 않는 것이 좋습니다.",
    "증상이 지속되면 병원 검진이 필요합니다.",
    "청결 유지와 자외선 차단이 중요합니다.",
    "약물치료나 연고 사용을 고려해볼 수 있습니다."
]

# 질문 조합식 템플릿
question_templates = [
    "이 사진의 피부 질환이 무엇인가요?",
    "피부에 이런 증상이 있는데 어떤 병일까요?",
    "이건 어떤 피부 질환으로 보이나요?",
    "이 사진 속 피부 상태를 진단해 주세요.",
    "피부가 이렇게 보이면 어떤 질환일 가능성이 있나요?",
    "이 사진에서 보이는 증상은 어떤 병인가요?",
    "이런 피부 변화는 어떤 문제 때문일까요?",
    "피부가 이렇게 변했는데 괜찮은 걸까요?",
    "사진을 보면 어떤 피부질환 같나요?",
    "피부가 가렵고 붉은데 이유가 뭘까요?"
]

label_groups = defaultdict(list)

# 데이터 로드
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)

        # ✅ 한글 정규화 (중요! '흑색종' → '흑색종')
        label_text = unicodedata.normalize("NFC", s["messages"][-1]["content"][0]["text"].lower().strip())
        label = None

        # 라벨 탐색 (한글+영문 대응)
        for disease, aliases in label_aliases.items():
            if any(alias in label_text for alias in aliases):
                label = disease
                break
        if not label:
            continue

        # 이미지 경로 추출
        user_contents = s["messages"][0]["content"]
        image_url = next((c.get("image_url") for c in user_contents if "image_url" in c), "img/unknown.jpg")

        # 랜덤 조합으로 문장 생성
        symptom = random.choice(symptom_parts[label])
        advice = random.choice(advice_parts)
        ending = random.choice(["보입니다.", "같아요.", "으로 추정됩니다.", "가능성이 있습니다.", "형태입니다."])
        full_text = f"이 이미지는 {label}{random.choice(['으로', '로'])} 보입니다. {symptom}. {advice} ({label}{ending})"

        # 새로운 질의/응답 샘플 생성
        new_question = random.choice(question_templates)
        new_sample = {
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image_url": image_url},
                    {"type": "text", "text": new_question}
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": full_text}
                ]}
            ]
        }

        label_groups[label].append(new_sample)

# 라벨별 100개 샘플링
selected_samples = []
label_stats = {}

for label, items in label_groups.items():
    n = min(100, len(items))
    sampled = random.sample(items, n)
    selected_samples.extend(sampled)
    label_stats[label] = (n, len(items))

# 저장
with open(output_path, "w", encoding="utf-8") as f:
    for s in selected_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"✅ 고급 다양화 + 라벨별 샘플링 완료 → {output_path}")
print(f"🎯 총 선택된 샘플 수: {len(selected_samples)}\n")

print("📊 라벨별 샘플링 결과:")
for label, (chosen, total) in label_stats.items():
    print(f" - {label}: {chosen}개 선택 (전체 {total}개 중)")


✅ 고급 다양화 + 라벨별 샘플링 완료 → /content/drive/MyDrive/Joleop Project/data/train/finetune_dataset_diverse_dynamic.jsonl
🎯 총 선택된 샘플 수: 800

📊 라벨별 샘플링 결과:
 - 멜라닌 세포모반: 100개 선택 (전체 7867개 중)
 - 흑색종: 100개 선택 (전체 3036개 중)
 - 지루각화증: 100개 선택 (전체 1625개 중)
 - 정상피부: 100개 선택 (전체 124개 중)
 - 여드름: 100개 선택 (전체 298개 중)
 - 아토피 피부염: 100개 선택 (전체 1119개 중)
 - 습진: 100개 선택 (전체 299개 중)
 - 각화증: 100개 선택 (전체 298개 중)


In [6]:
!pip install trl
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 39.1 MB/s eta 0:00:00


In [14]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [15]:
import json
import random
from collections import defaultdict
from typing import Any
from PIL import Image
import torch

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [16]:
# -----------------------------
# 1️⃣ MedGemma-4b 모델 로드
# -----------------------------
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# Check if GPU supports bfloat16
if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")

model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)

# Use right padding to avoid issues during training
processor.tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [27]:
import json
import random
from collections import defaultdict

jsonl_path = "/content/drive/MyDrive/Joleop Project/data/train/finetune_dataset_diverse_dynamic.jsonl"

random_state = 42
random.seed(random_state)

# 파일 읽기
with open(jsonl_path, "r", encoding="utf-8") as f:
    selected_samples = [json.loads(line) for line in f]

# -----------------------------
# 라벨별로 그룹화
# -----------------------------
label_groups = defaultdict(list)
for sample in selected_samples:
    try:
        # assistant의 첫 번째 text가 label임
        label = sample["messages"][-1]["content"][0]["text"].strip().lower()
        label_groups[label].append(sample)
    except Exception as e:
        print("⚠️ 라벨 추출 실패:", e)
        continue

# -----------------------------
# 라벨별로 100개씩 랜덤 선택
# -----------------------------
selected_by_label = []
for label, items in label_groups.items():
    n = min(100, len(items))
    selected_by_label.extend(random.sample(items, n))

# -----------------------------
# train / validation split (90% / 10%)
# -----------------------------
random.shuffle(selected_by_label)
split_idx = int(len(selected_by_label) * 0.9)
train_data = selected_by_label[:split_idx]
val_data = selected_by_label[split_idx:]

data = {
    "train": train_data,
    "validation": val_data
}

print(f"✅ 전체 샘플 수: {len(selected_samples)}")
print(f"🎯 라벨별 샘플링 후 총 수: {len(selected_by_label)}")
print(f"📘 Train: {len(train_data)}개")
print(f"📗 Validation: {len(val_data)}개")
print("📊 라벨 분포:")
for label, items in label_groups.items():
    print(f" - {label}: {len(items)}개 중 {min(100, len(items))}개 선택됨")


✅ 전체 샘플 수: 800
🎯 라벨별 샘플링 후 총 수: 800
📘 Train: 720개
📗 Validation: 80개
📊 라벨 분포:
 - 이 이미지는 멜라닌 세포모반으로 보입니다. 일반적인 모반 형태를 띱니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (멜라닌 세포모반가능성이 있습니다.): 3개 중 3개 선택됨
 - 이 이미지는 멜라닌 세포모반로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 손으로 자극하지 않는 것이 좋습니다. (멜라닌 세포모반가능성이 있습니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 청결 유지와 자외선 차단이 중요합니다. (멜라닌 세포모반형태입니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 보습제를 자주 바르면 도움이 됩니다. (멜라닌 세포모반보입니다.): 2개 중 2개 선택됨
 - 이 이미지는 멜라닌 세포모반로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 전문의의 진료를 권장드립니다. (멜라닌 세포모반보입니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 보습제를 자주 바르면 도움이 됩니다. (멜라닌 세포모반으로 추정됩니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반로 보입니다. 일반적인 모반 형태를 띱니다. 전문의의 진료를 권장드립니다. (멜라닌 세포모반보입니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 청결 유지와 자외선 차단이 중요합니다. (멜라닌 세포모반보입니다.): 1개 중 1개 선택됨
 - 이 이미지는 멜라닌 세포모반으로 보입니다. 색소가 고르게 분포되어 있습니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (멜라닌 세포모반가능성이 있습니다.): 2개 중 2개 선택됨
 

In [28]:
# -----------------------------
# 3️⃣ LoRA 설정
# -----------------------------
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)


In [29]:
# -----------------------------
# 4️⃣ 데이터 콜레이터
# -----------------------------
from PIL import Image

def collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        # JSONL 구조에 맞게 경로 불러오기
        image_path = example["messages"][0]["content"][0]["image_url"]
        img = Image.open(image_path).convert("RGB")
        images.append([img])

        texts.append(processor.apply_chat_template(
            example["messages"],
            add_generation_prompt=False,
            tokenize=False
        ).strip())

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch


In [30]:
# -----------------------------
# 5️⃣ 학습 파라미터 (SFTConfig)
# -----------------------------
from trl import SFTConfig

num_train_epochs = 1  # @param {type: "number"}
learning_rate = 2e-4  # @param {type: "number"}

args = SFTConfig(
    output_dir="/content/drive/MyDrive/Joleop Project/finetuned_models/medgemma-4b-it-sft-lora-crc100k-skin-disease",
    num_train_epochs=num_train_epochs,                       # Number of training epochs
    per_device_train_batch_size=1,                           # Batch size per device during training
    per_device_eval_batch_size=4,                            # Batch size per device during evaluation
    gradient_accumulation_steps=4,                           # Number of steps before performing a backward/update pass
    gradient_checkpointing=True,                             # Enable gradient checkpointing to reduce memory usage
    optim="adamw_torch_fused",                               # Use fused AdamW optimizer for better performance
    logging_steps=50,                                        # Number of steps between logs
    save_strategy="epoch",                                   # Save checkpoint every epoch
    eval_strategy="steps",                                   # Evaluate every `eval_steps`
    eval_steps=50,                                           # Number of steps between evaluations
    learning_rate=learning_rate,                             # Learning rate based on QLoRA paper
    bf16=True,                                               # Use bfloat16 precision
    max_grad_norm=0.3,                                       # Max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                                       # Warmup ratio based on QLoRA paper
    lr_scheduler_type="linear",                              # Use linear learning rate scheduler
    push_to_hub=True,                                        # Push model to Hub
    report_to="tensorboard",                                 # Report metrics to tensorboard
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Set gradient checkpointing to non-reentrant to avoid issues
    dataset_kwargs={"skip_prepare_dataset": True},           # Skip default dataset preparation to preprocess manually
    remove_unused_columns = False,                           # Columns are unused for training but needed for data collator
    label_names=["labels"],                                  # Input keys that correspond to the labels
)

In [31]:
from datasets import Dataset

# list → Hugging Face Dataset 변환
data["train"] = Dataset.from_list(data["train"])
data["validation"] = Dataset.from_list(data["validation"])

# 검증 subset 만들기
val_ds_subset = data["validation"].shuffle(seed=42).select(range(50))

# 학습용 변수 정의 (가독성용)
train_ds = data["train"]
val_ds = val_ds_subset


In [32]:
# -----------------------------
# 6️⃣ Trainer 구성 및 학습
# -----------------------------
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds_subset,  # 리스트가 아닌 Dataset
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [33]:
# 학습 시작
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.229700,0.209207,0.201561,64880.000000,0.914623
100,0.183700,0.168500,0.280098,129891.000000,0.926758
150,0.161600,0.151032,0.259243,194811.000000,0.923088


TrainOutput(global_step=180, training_loss=0.4630163457658556, metrics={'train_runtime': 1734.9216, 'train_samples_per_second': 0.415, 'train_steps_per_second': 0.104, 'total_flos': 6075974513253216.0, 'train_loss': 0.4630163457658556, 'entropy': 0.2997120313346386, 'num_tokens': 233663.0, 'mean_token_accuracy': 0.9185735861460368, 'epoch': 1.0})

In [34]:
trainer.save_model()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...90826.26f89daf39cf.3797.0: 100%|##########| 11.1kB / 11.1kB            

  ...disease/training_args.bin: 100%|##########| 6.48kB / 6.48kB            

  ...n-disease/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...in-disease/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...adapter_model.safetensors:   1%|          | 25.2MB / 2.84GB            

No files have been modified since last commit. Skipping to prevent empty commit.


In [35]:
del model
del trainer
torch.cuda.empty_cache()